## ___Updating the mycorrhizal states___
--------------------

In [1]:
!python --version

Python 3.13.9


The system cannot find the path specified.


In [3]:
import numpy as np
import pandas as pd

In [4]:
# https://datadryad.org/dataset/doi:10.5061/dryad.n8bm9
# THE SHEET "Original states data" HAS THE RAW DATA SCRAPED FROM PUBLICATIONS WITHOUT ANY INTEFERENCE FROM THE AUTHORS!!!!
maherali_original = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Original states data", skiprows=range(2),
                                  usecols=("Source", "Original name (Genus species)", "Raw state record from publication"))
maherali_original.rename(mapper={old: old.replace('(', '').replace(')', '').lower().replace(' ', '_') for old in maherali_original.columns}, axis=1, inplace=True) # column names have parentheses and spaces
# taxonomy columns in Maherali et. al. dataset has trailing spaces :(
maherali_original.loc[:, "original_name_genus_species"] = maherali_original.original_name_genus_species.str.strip()
maherali_original.loc[:, "raw_state_record_from_publication"] = maherali_original.raw_state_record_from_publication.str.strip()
maherali_original.drop_duplicates(subset=("original_name_genus_species", "raw_state_record_from_publication"), inplace=True)

# final_maherali = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Final list matched with phylo", skiprows=range(2))
# final_maherali.rename(mapper={old: old.lower().replace(' ', '_') for old in final_maherali.columns}, axis=1, inplace=True)
# final_maherali.genus_species = final_maherali.genus_species.str.strip().str.replace('_', ' ') # the sheet "Final list matched with phylo" has genus and specific epithets concatenated by under scores!

# in TRY, mycorrhiza type is trait id 7
try_myco = pd.read_csv(r"../../data/chapter2/TRY/mycorrhizal_states.txt", delimiter='\t', low_memory=False, encoding="latin1", usecols=["Dataset", "SpeciesName", "AccSpeciesName", "OrigValueStr",
                                "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"])
# unify the mycorrhizal state info
# 'ECTO', 'NM/AM', 'EC', 'EC/AM', 'AM', 'Ecto', 'Non',        'vesicular-arbuscular mycorrhiza', 'ectomycorrhiza', 'no', '0', 'Ph.th.end.', 'VAM', 'Ectomycorrhiza', 'E.ch.ect.', 'arbuscular',
# 'ec?', 'VA', 'ecto', 'Absent', 'non-ectomycorrhizal', 'ectomycorrhizal', 'Yes', 'No', 'EM', 'AMNM', 'NM', 'AM + EM', 'ERM', 'Ericoid', 'ECM'

MYCORRHIZAL_STATES_REPLACEMENTS = {
    "ECTO": "EcM",
    "Ecto": "EcM",
    "EC": "EcM",
    "ectomycorrhiza": "EcM",
    "Ectomycorrhiza": "EcM",
    "ecto": "EM",
    "ectomycorrhizal": "EcM",
    "ECM": "EcM",
    "vesicular-arbuscular mycorrhiza" : "AM",
    "VAM": "AM",
    "VA": "AM",
    "Non": "NM",
    "AMNM": "NM/AM",
    "Ericoid": "ErM",
    "ERM": "ErM",
    "AM + EM": "AM/EcM",
    "EC/AM": "AM/EcM",
    "Orchid": "OrM",
    "OrM": "OrM"
}

try_myco.loc[:, "OrigValueStr"] = try_myco.OrigValueStr.replace(MYCORRHIZAL_STATES_REPLACEMENTS)
try_myco = try_myco.query("OrigValueStr.isin(@MYCORRHIZAL_STATES_REPLACEMENTS.values())")

mycodb_v4 = pd.read_csv(r"../../data/chapter2/MycoDB_version4.csv", usecols=["PlantSpecies2018", "FUNGROUP", "MYCORRHIZAETYPE", "AM_single_genus", "EM_single_genus", 
                        "STERILIZED", "NONMYCOCONTROL", "NONMYCOCONTROL2"]).dropna(subset="PlantSpecies2018").drop_duplicates()
mycodb_v4.loc[:, "PlantSpecies2018"] = mycodb_v4.PlantSpecies2018.str.capitalize().str.replace('_', ' ')

# scrape the online only MycoDB metadata and serialize it to the disk
# req = Request(url=r"https://www.nature.com/articles/sdata201628/tables/2", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0"})
# with urlopen(req) as r:
#     soup = BeautifulSoup(r.read())
# 
# table = soup.find(name="table", attrs={"class": "data last-table"}) # locate the metadata table
# [th.text.strip() for th in table.find_all(name="th")] # column names
# mycodb_descriptions = [[td.text for td in tr.find_all(name="td")] for tr in table.find_all(name="tr")[1:]] # parse the rows
# 
# # create a dataframe using the parsed rows and column names and serialize it to the disk
# pd.DataFrame({ 
#     "Variable": [row[0] for row in mycodb_descriptions],
#     "Description": [row[1] for row in mycodb_descriptions],
#     "Variable Type (range)": [row[2] for row in mycodb_descriptions],
#     "Levels (#studies/level)": [row[3] for row in mycodb_descriptions],
# }).to_csv(r"../data/chapter2/MycoDB_version4_metadata.csv", index=False)

mycodb_v4_meta = pd.read_csv(r"../../data/chapter2/MycoDB_version4_metadata.csv") # read in the pre scraped & serialized .csv file

subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

# this only has genus level mycorrhizal types
fungalroot = pd.read_csv(r"../../data/chapter2/FungalRoot/FungalRoot_cleaned.csv", low_memory=False, encoding="utf-8")
fungalroot.loc[:, "species"] = fungalroot.species.str.strip()

# even though we had missing data for photosynthetic pathways and mycorrhizal states in the records that had data for the 4 chosen traits in FRED, FRED could still have info for the categorical traits in other records 
# that were filtered out due to not having all the 4 trait values?????
# do a re-lookup!
fred_myco = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1",
                        usecols=("F01286", "F01287", "F00645", "F00004")).dropna(subset=("F01286", "F01287", "F00645")).drop_duplicates()
# concatenate the genus name and specific epithet to introduce a column for binominal name
fred_myco.insert(loc=0, column="binominal", value=fred_myco.F01286.str.strip().str.capitalize() + ' ' + fred_myco.F01287.str.strip().str.lower())

In [5]:
# we are not just looking to fill the missing mycorrhizal state info here!
# we could potentially find and reconcile conflicts between information in FRED and other databases!
pd.merge(left=subset_categorical.loc[:, ["binominal", "F00645", "F00004"]], left_on="binominal", right=maherali_original, right_on="original_name_genus_species", how="inner")

,binominal,F00645,F00004,source,original_name_genus_species,raw_state_record_from_publication
0,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Wang&Qiu2006,Populus trichocarpa,EM
1,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Hempel et al. 2013,Populus trichocarpa,AM+EM
2,Populus tremula,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Wang&Qiu2006,Populus tremula,AM + EM
3,Populus tremula,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Akhmetzhanova et al. 2012,Populus tremula,EM
4,Fraxinus excelsior,AM,"Kubisch P, Hertel D, Leuschner C. 2015. Do ect...",Wang&Qiu2006,Fraxinus excelsior,AM AM + EM
...,...,...,...,...,...,...
92,Magnolia kobus,NaN,Valverde et al (unpublished),Akhmetzhanova et al. 2012,Magnolia kobus,AM
93,Magnolia tripetala,NaN,Valverde et al (unpublished),Akhmetzhanova et al. 2012,Magnolia tripetala,AM
94,Magnolia virginiana,NaN,Valverde et al (unpublished),Akhmetzhanova et al. 2012,Magnolia virginiana,AM
95,Populus deltoides,NaN,Valverde et al (unpublished),Wang&Qiu2006,Populus deltoides,AM


In [48]:
# same for FungalRoot too. 
# instead of just populating the empty cells, look for conflicts between data that exists in FRED and FungalRoot!
pd.merge(left=subset_categorical.loc[:, ["binominal", "F00645", "F00004"]], left_on="binominal", right=fungalroot, right_on="species", how="inner")

,binominal,F00645,F00004,original_reference,original_ref_checked,non_original_reference,habitat_naturality,biome,species,mycorrhiza type,remark_mycorrhiza_ type,curator_remark_1_name,curator_remark_1_comment,curator_remark_2_name,curator _remark_2_comment,curator_remark_3_name,curator_remark_3_comment,other_remarks,consensus_mycorrhizal_state
0,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Akhmetzhanova, A. A., Soudzilovskaia, N. A., O...",NaN,NaN,botanical garden,NaN,Populus trichocarpa,EcM; others not addressed,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
1,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Schultz, R. C., Isebrands, J. G., & Kormanik, ...",NaN,NaN,NaN,NaN,Populus trichocarpa,EcM; no others,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
2,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Baum, C., & Makeschin, F. (2000). Effects of n...",y,"Wang, B., & Qiu, Y. L. (2006). Phylogenetic di...",NaN,NaN,Populus trichocarpa,EcM; others not addressed,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
3,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Trappe, J. (1962). Bot. Rev., 23, 528-606.",y,"Harley, J. L., & Harley, E. L. (1987). A check...",NaN,NaN,Populus trichocarpa,EcM; others not addressed,NaN,Leho Tedersoo,in this study 26% reports conflict with predic...,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
4,Populus trichocarpa,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Harley, J. L., & Brierley, J. K. (1954). The u...",y,"Harley, J. L., & Harley, E. L. (1987). A check...",NaN,NaN,Populus trichocarpa,EcM; others not addressed,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,Ulmus americana,NaN,Valverde et al (unpublished),"Brundrett, M., Murase, G., & Kendrick, B. (199...",NaN,NaN,natural,NaN,Ulmus americana,AM; no others,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AM
898,Ulmus americana,NaN,Valverde et al (unpublished),"McDougall, W. B. (1914). On the mycorrhizas of...",NaN,NaN,NaN,NaN,Ulmus americana,non-mycorrhizal (checked for all types),NaN,Leho Tedersoo,probably incorrect (this genus is AM; in this ...,NaN,NaN,NaN,NaN,NaN,AM
899,Ulmus americana,NaN,Valverde et al (unpublished),"Thomas Jr, W. D. (1943). Mycorrhizae associate...",NaN,NaN,NaN,NaN,Ulmus americana,EcM; no others,NaN,Leho Tedersoo,incorrect report (this genus is AM; in this st...,NaN,NaN,Laura M. Suz,not ECM,NaN,AM
900,Ulmus americana,NaN,Valverde et al (unpublished),"Vozzo, J. A., & Hacskaylo, E. (1964). Anatomy ...",NaN,NaN,NaN,NaN,Ulmus americana,AM; no others,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AM


In [46]:
# repeat the same process with TRY
pd.merge(left=subset_categorical, left_on="binominal", right=try_myco, right_on="AccSpeciesName", how="inner")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004,Dataset,SpeciesName,AccSpeciesName,TraitID,OrigValueStr
0,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Abisko & Sheffield Database,Populus tremula,Populus tremula,7.0,EcM
1,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Global 15N Database,Populus tremula,Populus tremula,7.0,NM
2,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Global 15N Database,Populus tremula,Populus tremula,7.0,NM
3,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Global 15N Database,Populus tremula,Populus tremula,7.0,NM
4,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Global 15N Database,Populus tremula,Populus tremula,7.0,NM
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7772,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),FRED - Fine Root Ecology Database,Ulmus americana,Ulmus americana,7.0,AM
7773,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),FRED - Fine Root Ecology Database,Ulmus americana,Ulmus americana,7.0,AM
7774,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),FRED - Fine Root Ecology Database,Ulmus americana,Ulmus americana,7.0,AM
7775,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),Independent evolutionary changes in fine-root ...,Ulmus americana,Ulmus americana,7.0,AM


In [47]:
# again, with FRED
# cool :)
pd.merge(left=subset_categorical.loc[:, ["binominal", "F00645", "F00004"]].query("F00645.isna()"), left_on="binominal", right=fred_myco, right_on="binominal", suffixes=(None, "_full"), how="inner")

,binominal,F00645,F00004,F00004_full,F01286,F01287,F00645_full
0,Populus tremula,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Sun T, Mao Z, Han Y. 2013. Slow decomposition ...",Populus,tremula,EM
1,Populus tremula,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",Populus,tremula,EM
2,Populus tremula,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Populus,tremula,EM
3,Cryptocarya chinensis,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...","Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",Cryptocarya,chinensis,AM
4,Cryptocarya chinensis,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...","Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Cryptocarya,chinensis,AM
...,...,...,...,...,...,...,...
140,Syringa reticulata,NaN,Valverde et al (unpublished),"Chen W, Zeng H, Eissenstat DM, Guo D. 2013. Va...",Syringa,reticulata,AM + EM
141,Syringa reticulata,NaN,Valverde et al (unpublished),"Guo D, Xia M, Wei X, Chang W, Liu Y, Wang Z. 2...",Syringa,reticulata,AM + EM
142,Syringa reticulata,NaN,Valverde et al (unpublished),"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Syringa,reticulata,AM
143,Syringa reticulata,NaN,Valverde et al (unpublished),"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",Syringa,reticulata,AM


In [20]:
 subset_categorical.F00645.unique()

array([nan, 'AM', 'EM', 'AM + EM', 'NM', 'ErM'], dtype=object)

In [23]:
try_myco.OrigValueStr.unique()

array(['NM/AM', 'EM', 'AM/EM', 'ER', 'AM', 'NM', 'OM'], dtype=object)

In [22]:
pd.merge(left=subset_categorical, left_on="binominal", right=try_myco, right_on="AccSpeciesName", how="inner").drop_duplicates(subset=["binominal", "OrigValueStr"]).OrigValueStr.unique()

array(['EM', 'NM', 'AM', 'NM/AM', 'AM/EM', 'ER'], dtype=object)

In [72]:
# instead of making a mess with merging datasets in pairs, filter out the subsets that contain data for the species of interest
species_of_interest = subset_categorical.binominal.drop_duplicates()
species_of_interest

0         Populus trichocarpa
1             Populus tremula
2            Altingia obovata
3       Cryptocarya chinensis
4      Elaeocarpus sylvestris
                ...          
207              Quercus alba
208             Quercus rubra
210       Pleioblastus amarus
234            Nyssa aquatica
235       Platanus acerifolia
Name: binominal, Length: 203, dtype: object

In [68]:
fred_myco.query("binominal.isin(@species_of_interest)").drop_duplicates().to_csv(r"../../data/chapter2/extracts/fred.csv", index=False)
try_myco.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates().to_csv(r"../../data/chapter2/extracts/try.csv", index=False)
maherali_original.query("original_name_genus_species.isin(@species_of_interest)").to_csv(r"../../data/chapter2/extracts/maherali.csv", index=False)
fungalroot.query("species.isin(@species_of_interest)").to_csv(r"../../data/chapter2/extracts/fungalroot.csv", index=False)